# Datasets preparation

In [1]:
import shutil
import os
from tqdm import tqdm
import copy
import random
from sklearn.model_selection import StratifiedKFold

## UCF101

In [2]:
ucf101_path = 'Z:/Xavi/UCF101/UCF101'
ucf101_index = 'Z:/Xavi/UCF101/Class Index.txt'
ucf101_label_map = 'Z:/Xavi/UCF101/label_map_UCF101.txt'
ucf101_videos_path = 'Z:/Xavi/UCF101/videos'
ucf101_videos_file = 'Z:/Xavi/UCF101/UCF101-videos.txt'
train_file = 'Z:/Xavi/UCF101/UCF101_train_video.txt'
test_file = 'Z:/Xavi/UCF101/UCF101_test_video.txt'
val_root_name = 'Z:/Xavi/UCF101/UCF101_'

### Copy all videos to one folder

In [ ]:
# Get all the videos in UCF-101 folder
ucf101_videos = []
for root, dirs, files in os.walk(ucf101_path):
    for file in files:
        if file.endswith('.avi'):
            ucf101_videos.append(os.path.join(root, file))

# Create folder for UCF-101 videos
if not os.path.exists(ucf101_videos_path):
    os.makedirs(ucf101_videos_path)

# Copy videos
for video in tqdm(ucf101_videos):
    shutil.copy(video, ucf101_videos_path)

### Create label map

In [7]:
# Load file line by line and write only class name (2nd column)
with open(ucf101_index, 'r') as f_in, open(ucf101_label_map, 'w') as f_out:
    for line in f_in:
        f_out.write(line.split(' ')[1])

### Create labels file

In [3]:
# Load class index into dictionary
class_index = {}
with open(ucf101_index, 'r') as f:
    for line in f:
        class_index[line.split()[1]] = int(line.split()[0])-1

# Iterate over folders in UCF-101, and videos in each folder. They are the classe and video names.
# Then write file name with label to a text file
with open(ucf101_videos_file, 'w') as f:
    for root, dirs, files in tqdm(os.walk(ucf101_path)):
        for file in files:
            if file.endswith('.avi'):
                class_name = root.split('\\')[-1]
                f.write(file + ' ' + str(class_index[class_name]) + '\n')
 

102it [00:00, 2689.52it/s]


### Split into train and test sets

There are videos splitted into various clips, so all clips pertaining to the same video should be included into the same set, to avoid overfitting.

In [8]:
# Load video list into dictionary of dictionaries of lists. Key is video class, and value is a dictionary with base video as key name and value as list of video names.
videos = {}
with open(ucf101_videos_file, 'r') as f:
    for line in f:
        video_name = line.split()[0]
        class_name = line.split()[1]
        base_video_name = line.split('_')[-2]

        # Add video name to list of videos for base video of a class
        if class_name not in videos:
            videos[class_name] = {}
        if base_video_name not in videos[class_name]:
            videos[class_name][base_video_name] = []
        videos[class_name][base_video_name].append(video_name)

# Remove videos and leave only a dictionary of lists, where key is class name and value is list of video base names
video_groups = copy.deepcopy(videos)
for class_name in videos:
    video_groups[class_name] = list(video_groups[class_name].keys())

# Perform random stratified train-test splits (80-20)
train = {}
test = {}
for class_name in video_groups:
    train[class_name] = []
    test[class_name] = []

    # Calculate number of videos for train and test
    n_videos = len(video_groups[class_name])
    n_train = int(n_videos * 0.8)
    n_test = n_videos - n_train

    # Shuffle videos
    random.shuffle(video_groups[class_name])
    
    # Split videos
    train[class_name] = video_groups[class_name][:n_train]
    test[class_name] = video_groups[class_name][n_train:]

# Add videos to train and test dictionaries
train_videos = {}
test_videos = {}
for class_name in train:
    train_videos[class_name] = []
    test_videos[class_name] = []

    for base_video_name in train[class_name]:
        train_videos[class_name] += videos[class_name][base_video_name]

    for base_video_name in test[class_name]:
        test_videos[class_name] += videos[class_name][base_video_name]

# Write train and test videos to text files
with open(train_file, 'w') as f:
    for class_name in train_videos:
        for video_name in train_videos[class_name]:
            f.write(video_name + ' ' + class_name + '\n')
with open(test_file, 'w') as f:
    for class_name in test_videos:
        for video_name in test_videos[class_name]:
            f.write(video_name + ' ' + class_name + '\n')

## Split for stratified k-fold cross validation

There are videos splitted into various clips, so all clips pertaining to the same video should be included into the same set, to avoid overfitting.

In [4]:
# Load video list into dictionary of dictionaries of lists. Key is video class, and value is a dictionary with base video as key name and value as list of video names.
videos = {}
with open(ucf101_videos_file, 'r') as f:
    for line in f:
        video_name = line.split()[0]
        class_name = line.split()[1]
        base_video_name = line.split('_')[-2]

        # Add video name to list of videos for base video of a class
        if class_name not in videos:
            videos[class_name] = {}
        if base_video_name not in videos[class_name]:
            videos[class_name][base_video_name] = []
        videos[class_name][base_video_name].append(video_name)

# Get all video names and class names
x = []
y = []
for class_name in videos:
    x += videos[class_name].keys()
    y += [class_name] * len(videos[class_name])

# Split video_groups into 5 folds using sklearn StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=777)

# Iterate over folds, split videos into train and test, shuffle them and write each fold to a train and test txt files
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    
    # Training set
    temp = []
    for index in train_index:
        class_name = y[index]
        for video_name in videos[class_name][x[index]]:
            temp.append(video_name + ' ' + class_name + '\n')
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_train_video.txt', 'w') as f_train:
        for line in temp:
            f_train.write(line)

    # Testing set
    temp = []
    for index in test_index:
        class_name = y[index]
        for video_name in videos[class_name][x[index]]:
            temp.append(video_name + ' ' + class_name + '\n')
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_test_video.txt', 'w') as f_test:
        for line in temp:
            f_test.write(line)

## EtriActivity3D

In [5]:
etri_path = 'Z:/Xavi/EtriActivity3D'
etri_index = 'Z:/Xavi/EtriActivity3D/class_names.txt'
etri_label_map = 'Z:/Xavi/EtriActivity3D/label_map_EtriActivity3D.txt'
etri_videos_path = 'Z:/Xavi/EtriActivity3D/videos'
etri_videos_file = 'Z:/Xavi/EtriActivity3D/EtriActivity3D-videos.txt'
val_root_name = 'Z:/Xavi/EtriActivity3D/EtriActivity3D_'

### By video

All perspectives of the same action, person and place count as one video:

In [6]:
# Load video list into dictionary of dictionaries of lists. Key is video class, and value is a dictionary with base video as key name and value as list of video names.
videos = {}
with open(etri_videos_file, 'r') as f:
    for line in f:
        video_name = line.split()[0]
        class_name = line.split()[1]
        base_video_name = line.split('_')[1]+'_'+line.split('_')[2]

        # Add video name to list of videos for base video of a class
        if class_name not in videos:
            videos[class_name] = {}
        if base_video_name not in videos[class_name]:
            videos[class_name][base_video_name] = []
        videos[class_name][base_video_name].append(video_name)

# Get all video names and class names
x = []
y = []
for class_name in videos:
    x += videos[class_name].keys()
    y += [class_name] * len(videos[class_name])

# Split video_groups into 5 folds using sklearn StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=777)

# Iterate over folds, split videos into train and test, shuffle them and write each fold to a train and test txt files
for i, (train_index, test_index) in enumerate(skf.split(x, y)):
    
    # Training set
    temp = []
    for index in train_index:
        class_name = y[index]
        for video_name in videos[class_name][x[index]]:
            temp.append(video_name + ' ' + class_name + '\n')
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_train_video.txt', 'w') as f_train:
        for line in temp:
            f_train.write(line)

    # Testing set
    temp = []
    for index in test_index:
        class_name = y[index]
        for video_name in videos[class_name][x[index]]:
            temp.append(video_name + ' ' + class_name + '\n')
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_test_video.txt', 'w') as f_test:
        for line in temp:
            f_test.write(line)

### By user

Users splitted in 5. Proportion of young and elderly is maintained (50 and 50).

In [11]:
# Load video list into dictionary, where key is person and value is tuple of video name and class name
videos = {}
with open(etri_videos_file, 'r') as f:
    for line in f:
        video_name = line.split()[0]
        class_name = line.split()[1]
        base_video_name = line.split('_')[1]

        # Add video name to list of videos for base video of a class
        if base_video_name not in videos:
            videos[base_video_name] = []
        videos[base_video_name].append((video_name, class_name))

# Initialize kfold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=777)
elderly_indexes = list(range(1, 51))
young_indexes = list(range(51, 101))
temp_train = {}
temp_test = {}

# Elderly users: 1-50
for i, (train_index, test_index) in enumerate(skf.split(elderly_indexes, [0] * 50)):
    
    # Training set
    temp = []
    for index in train_index:
        person_key = 'P'+str(elderly_indexes[index]).zfill(3)
        for video_name, class_name in videos[person_key]:
            temp.append(video_name + ' ' + class_name + '\n')
    if i not in temp_train:
        temp_train[i] = []
    temp_train[i] += temp

    # Testing set
    temp = []
    for index in test_index:
        person_key = 'P'+str(elderly_indexes[index]).zfill(3)
        for video_name, class_name in videos[person_key]:
            temp.append(video_name + ' ' + class_name + '\n')
    if i not in temp_test:
        temp_test[i] = []
    temp_test[i] += temp

# Young users: 51-100
for i, (train_index, test_index) in enumerate(skf.split(young_indexes, [0] * 50)):
    
    # Training set
    temp = []
    for index in train_index:
        person_key = 'P'+str(young_indexes[index]).zfill(3)
        for video_name, class_name in videos[person_key]:
            temp.append(video_name + ' ' + class_name + '\n')
    if i not in temp_train:
        temp_train[i] = []
    temp_train[i] += temp

    # Testing set
    temp = []
    for index in test_index:
        person_key = 'P'+str(young_indexes[index]).zfill(3)
        for video_name, class_name in videos[person_key]:
            temp.append(video_name + ' ' + class_name + '\n')
    if i not in temp_test:
        temp_test[i] = []
    temp_test[i] += temp
    
# Write train and test videos to text files
for i, temp in temp_train.items():
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_train_video.txt', 'w') as f_train:
        for line in temp:
            f_train.write(line)
for i, temp in temp_test.items():
    random.shuffle(temp)
    with open(val_root_name + str(i+1) + '_test_video.txt', 'w') as f_test:
        for line in temp:
            f_test.write(line)